In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
%python
dbutils.widgets.text("init_load_flag", '1')  # or "1", whichever is correct
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

In [0]:
init_load_flag

1

### **Read Data**

In [0]:
df=spark.sql("""
             select * from databricks_cata.silver.customer_silver""")

## **Removig Duplicate**

In [0]:
df=df.dropDuplicates(subset=['customer_id'])

## **Dividing New VS Old records**

In [0]:
if init_load_flag==0:

    df_old=spark.sql('''
                     select DimCustomerKey,customer_id,create_date,update_date from databricks_cata.gold.DimCustomer''')
    
else:

    df_old=spark.sql(''' 
                    select 0 DimCustomerKey,0 customer_id,0 create_date,0 update_date from databricks_cata.silver.customer_silver where 1=0 ''')

In [0]:
df_old.display()

DimCustomerKey,customer_id,create_date,update_date


**Renaming columns of df_old**

In [0]:
df_old=df_old.withColumnRenamed("DimCustomerKey","old_DimCustomerKey")\
    .withColumnRenamed("customer_id","old_customer_id")\
    .withColumnRenamed("create_date","old_create_date")\
    .withColumnRenamed("update_date","old_update_date")

## **Applying the join with old recordes**

In [0]:
df_join=df.join(df_old,df.customer_id==df_old.old_customer_id,'left')

In [0]:
df_join.display()

customer_id,email,city,state,domain,Full_name,old_DimCustomerKey,old_customer_id,old_create_date,old_update_date
C01991,craigbarajas@williamson.com,Sylviafort,MS,williamson.com,Sonia Matthews,null,null,null,null
C01992,pcross@hotmail.com,New Anthonybury,MT,hotmail.com,Andrew Turner,null,null,null,null
C01993,careyjohn@ortiz.com,Port Larrymouth,TN,ortiz.com,Daniel Flynn,null,null,null,null
C01994,barkertaylor@hampton.info,North Ericshire,NJ,hampton.info,Nicole Griffin,null,null,null,null
C01995,millerjodi@hotmail.com,Wolffort,FL,hotmail.com,Vincent Long,null,null,null,null
C01996,alan72@salas.com,Villarrealtown,WV,salas.com,David Warren,null,null,null,null
C01997,raydana@sanders.com,New Gabriel,NM,sanders.com,Amber Lawson,null,null,null,null
C01998,beckyjones@hotmail.com,Davidland,NH,hotmail.com,Ivan Jenkins,null,null,null,null
C01999,brandondiaz@mcdowell.biz,Stephaniechester,TX,mcdowell.biz,Kathleen Hodges,null,null,null,null
C02000,mcdowellkatie@yahoo.com,West Dianechester,DE,yahoo.com,Julie Smith,null,null,null,null


**Seperating new vs old records**

In [0]:
df_new=df_join.filter(df_join['old_DimCustomerKey'].isNull()) 

In [0]:
df_old=df_join.filter(df_join['old_DimCustomerKey'].isNotNull()) 

**Preparing df_old**

In [0]:
# Dropping all the columns which are not requried
df_old=df_old.drop('old_customer_id','old_update_date')

#renaming "old_DimCustomerKey" column to "DimCustomerKey"
df_old=df_old.withColumnRenamed("old_DimCustomerKey","DimCustomerKey")

# Renaming create date column "old_create_date" to "create_date"
df_old=df_old.withColumnRenamed("old_create_date","create_date")
df_old=df_old.withColumn("create_date",to_timestamp(col("create_date")))
#Recreationg "Update_date"
df_old=df_old.withColumn("update_date",current_timestamp())


In [0]:
df_old.display()

customer_id,email,city,state,domain,Full_name,DimCustomerKey,create_date,update_date


**Preparing df_new**

In [0]:
# Dropping all the columns which are not requried
df_new=df_new.drop('old_DimCustomerKey','old_customer_id','old_update_date','old_create_date')


#Recreationg "Update_date" and "current_date" column with current timestamp
df_new=df_new.withColumn("update_date",current_timestamp())
df_new=df_new.withColumn("create_date",current_timestamp())



In [0]:
df_new.display()

customer_id,email,city,state,domain,Full_name,update_date,create_date
C01991,craigbarajas@williamson.com,Sylviafort,MS,williamson.com,Sonia Matthews,2026-08-08T15:11:25.406Z,2026-08-08T15:11:25.406Z
C01992,pcross@hotmail.com,New Anthonybury,MT,hotmail.com,Andrew Turner,2026-08-08T15:11:25.406Z,2026-08-08T15:11:25.406Z
C01993,careyjohn@ortiz.com,Port Larrymouth,TN,ortiz.com,Daniel Flynn,2026-08-08T15:11:25.406Z,2026-08-08T15:11:25.406Z
C01994,barkertaylor@hampton.info,North Ericshire,NJ,hampton.info,Nicole Griffin,2026-08-08T15:11:25.406Z,2026-08-08T15:11:25.406Z
C01995,millerjodi@hotmail.com,Wolffort,FL,hotmail.com,Vincent Long,2026-08-08T15:11:25.406Z,2026-08-08T15:11:25.406Z
C01996,alan72@salas.com,Villarrealtown,WV,salas.com,David Warren,2026-08-08T15:11:25.406Z,2026-08-08T15:11:25.406Z
C01997,raydana@sanders.com,New Gabriel,NM,sanders.com,Amber Lawson,2026-08-08T15:11:25.406Z,2026-08-08T15:11:25.406Z
C01998,beckyjones@hotmail.com,Davidland,NH,hotmail.com,Ivan Jenkins,2026-08-08T15:11:25.406Z,2026-08-08T15:11:25.406Z
C01999,brandondiaz@mcdowell.biz,Stephaniechester,TX,mcdowell.biz,Kathleen Hodges,2026-08-08T15:11:25.406Z,2026-08-08T15:11:25.406Z
C02000,mcdowellkatie@yahoo.com,West Dianechester,DE,yahoo.com,Julie Smith,2026-08-08T15:11:25.406Z,2026-08-08T15:11:25.406Z


## **Surrogate key - from 1**

In [0]:
df_new=df_new.withColumn("DimCustomerKey",monotonically_increasing_id()+lit(1))


In [0]:
df_new.display()

customer_id,email,city,state,domain,Full_name,update_date,create_date,DimCustomerKey
C01991,craigbarajas@williamson.com,Sylviafort,MS,williamson.com,Sonia Matthews,2026-08-08T15:11:27.616Z,2026-08-08T15:11:27.616Z,1
C01992,pcross@hotmail.com,New Anthonybury,MT,hotmail.com,Andrew Turner,2026-08-08T15:11:27.616Z,2026-08-08T15:11:27.616Z,2
C01993,careyjohn@ortiz.com,Port Larrymouth,TN,ortiz.com,Daniel Flynn,2026-08-08T15:11:27.616Z,2026-08-08T15:11:27.616Z,3
C01994,barkertaylor@hampton.info,North Ericshire,NJ,hampton.info,Nicole Griffin,2026-08-08T15:11:27.616Z,2026-08-08T15:11:27.616Z,4
C01995,millerjodi@hotmail.com,Wolffort,FL,hotmail.com,Vincent Long,2026-08-08T15:11:27.616Z,2026-08-08T15:11:27.616Z,5
C01996,alan72@salas.com,Villarrealtown,WV,salas.com,David Warren,2026-08-08T15:11:27.616Z,2026-08-08T15:11:27.616Z,6
C01997,raydana@sanders.com,New Gabriel,NM,sanders.com,Amber Lawson,2026-08-08T15:11:27.616Z,2026-08-08T15:11:27.616Z,7
C01998,beckyjones@hotmail.com,Davidland,NH,hotmail.com,Ivan Jenkins,2026-08-08T15:11:27.616Z,2026-08-08T15:11:27.616Z,8
C01999,brandondiaz@mcdowell.biz,Stephaniechester,TX,mcdowell.biz,Kathleen Hodges,2026-08-08T15:11:27.616Z,2026-08-08T15:11:27.616Z,9
C02000,mcdowellkatie@yahoo.com,West Dianechester,DE,yahoo.com,Julie Smith,2026-08-08T15:11:27.616Z,2026-08-08T15:11:27.616Z,10


**Adding max Surrogate Key**

In [0]:
if init_load_flag==1:
    max_surrogate_key=0
else:
    df_maxsur=spark.sql('select max(DimCustomerKey) as max_surrogate_key from databricks_cata.gold.DimCustomers')
    # Convertiong df_maxsur to max_surrogate_key variable
    max_surrogate_key=df_maxsur.collect()[0]['max_surrogate_key']

In [0]:
df_new=df_new.withColumn("DimCustomerKey",lit(max_surrogate_key)+col("DimCustomerKey"))

**Unio of df_old and df_new**

In [0]:
df_final=df_new.unionByName(df_old)

In [0]:
df_final.display()

customer_id,email,city,state,domain,Full_name,update_date,create_date,DimCustomerKey
C01991,craigbarajas@williamson.com,Sylviafort,MS,williamson.com,Sonia Matthews,2026-08-08T15:11:30.618Z,2026-08-08T15:11:30.618Z,1
C01992,pcross@hotmail.com,New Anthonybury,MT,hotmail.com,Andrew Turner,2026-08-08T15:11:30.618Z,2026-08-08T15:11:30.618Z,2
C01993,careyjohn@ortiz.com,Port Larrymouth,TN,ortiz.com,Daniel Flynn,2026-08-08T15:11:30.618Z,2026-08-08T15:11:30.618Z,3
C01994,barkertaylor@hampton.info,North Ericshire,NJ,hampton.info,Nicole Griffin,2026-08-08T15:11:30.618Z,2026-08-08T15:11:30.618Z,4
C01995,millerjodi@hotmail.com,Wolffort,FL,hotmail.com,Vincent Long,2026-08-08T15:11:30.618Z,2026-08-08T15:11:30.618Z,5
C01996,alan72@salas.com,Villarrealtown,WV,salas.com,David Warren,2026-08-08T15:11:30.618Z,2026-08-08T15:11:30.618Z,6
C01997,raydana@sanders.com,New Gabriel,NM,sanders.com,Amber Lawson,2026-08-08T15:11:30.618Z,2026-08-08T15:11:30.618Z,7
C01998,beckyjones@hotmail.com,Davidland,NH,hotmail.com,Ivan Jenkins,2026-08-08T15:11:30.618Z,2026-08-08T15:11:30.618Z,8
C01999,brandondiaz@mcdowell.biz,Stephaniechester,TX,mcdowell.biz,Kathleen Hodges,2026-08-08T15:11:30.618Z,2026-08-08T15:11:30.618Z,9
C02000,mcdowellkatie@yahoo.com,West Dianechester,DE,yahoo.com,Julie Smith,2026-08-08T15:11:30.618Z,2026-08-08T15:11:30.618Z,10


## **SCD Type 1**

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("databricks_cata.gold.DimCustomers"):
    dlt_obj=DeltaTable.forPath(spark,"abfss://gold@databricksetese2.dfs.core.windows.net/DimCustomers")
   
    dlt_obj.alias("trg").merge(df_final.alias("src"),"trg.DimCustomerKey=src.DimCustomerKey").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_final.write.mode("overwrite").option("path","abfss://gold@databricksetese2.dfs.core.windows.net/DimCustomers").saveAsTable("databricks_cata.gold.DimCustomers")